# Phishing URL Detection Using Machine Learning

## Notebook 02 — Data Preprocessing

---

## 1. Introduction

Data preprocessing is an essential stage in a Machine Learning project.

The purpose of preprocessing is to convert the original dataset into a suitable format for Machine Learning algorithms while preserving the important information contained in the data.

The PhiUSIIL Phishing URL (Website) Dataset contains URL, domain, webpage, and other website-related characteristics. These features have different data types, ranges, distributions, and cardinalities.

Therefore, appropriate preprocessing decisions are required before model training.

---

## 2. Objectives

The main objectives of this notebook are:

1. Load the original dataset.
2. Verify the dataset structure.
3. Separate input features and target variable.
4. Remove metadata columns.
5. Identify numerical and categorical features.
6. Examine the URL-level duplication identified during dataset analysis.
7. Prepare the data for train-test splitting.
8. Prevent data leakage.
9. Apply appropriate categorical encoding.
10. Apply numerical feature scaling where required.
11. Verify the final processed feature matrices.
12. Save the preprocessing information required for later Machine Learning stages.

---

## 3. Dataset

The dataset used is the:

**PhiUSIIL Phishing URL (Website) Dataset**

The original dataset contains:

- **235,795 records**
- **56 columns**
- **54 documented predictive features**
- **1 metadata column: `FILENAME`**
- **1 target variable: `label`**

The target variable represents two classes:

- `0` — Phishing
- `1` — Legitimate

---

## 4. Preprocessing Pipeline

The preprocessing workflow will follow this general sequence:

```text
Original Dataset
       |
       v
Load Dataset
       |
       v
Initial Verification
       |
       v
Separate Target Variable
       |
       v
Remove Metadata
       |
       v
Identify Feature Types
       |
       v
Handle Text/Categorical Features
       |
       v
Train-Test Split
       |
       v
Fit Preprocessing on Training Data
       |
       v
Transform Training and Testing Data
       |
       v
Final Processed Features

In [1]:
# Importing libraries required for data preprocessing

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("All preprocessing libraries imported successfully.")

All preprocessing libraries imported successfully.


In [2]:
# Loading the dataset for preprocessing

dataset_path = "../dataset/PhiUSIIL_Phishing_URL_Dataset.csv"

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Number of records:", len(df))
print("Number of columns:", len(df.columns))

Dataset loaded successfully!
Dataset shape: (235795, 56)
Number of records: 235795
Number of columns: 56


## 2. Initial Dataset Inspection

Before performing any preprocessing operations, the dataset is inspected to understand its basic structure and verify that it has been loaded correctly.

The following aspects will be checked:

- Number of records and features
- Column names
- Data types of all columns
- Target variable
- Metadata columns
- Numerical and categorical/text columns

This initial inspection is important because preprocessing decisions such as encoding, scaling, feature removal, and missing-value treatment depend on the structure of the dataset.

The target variable in this project is `label`, where:

- `0` represents a phishing URL
- `1` represents a legitimate URL

The `FILENAME` column is treated as metadata and will not be used as a predictive feature.

In [3]:
# Inspecting dataset columns and data types

print("COLUMN INFORMATION")
print("=" * 60)

column_info = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

display(column_info)

print("\nDATA TYPE SUMMARY")
print("=" * 60)
print(df.dtypes.value_counts())

COLUMN INFORMATION


,Column Name,Data Type
0,FILENAME,str
1,URL,str
2,URLLength,int64
3,Domain,str
4,DomainLength,int64
5,IsDomainIP,int64
6,TLD,str
7,URLSimilarityIndex,float64
8,CharContinuationRate,float64
9,TLDLegitimateProb,float64



DATA TYPE SUMMARY
int64      41
float64    10
str         5
Name: count, dtype: int64


### Interpretation of Column Structure

The dataset contains both numerical and non-numerical attributes.

The numerical features represent measurable characteristics extracted from URLs and their associated webpages, such as the number of dots, digits, special characters, links, scripts, images, and other structural properties.

The non-numerical columns include URL-related textual information such as the complete URL, domain, top-level domain (TLD), webpage title, and filename.

For machine learning:

- Numerical features can be used after appropriate preprocessing and scaling.
- Categorical features such as `TLD` may require encoding.
- High-cardinality text fields such as `URL`, `Domain`, and `Title` require careful handling to avoid unnecessary dimensionality and data leakage.
- `FILENAME` is metadata and will be excluded from the predictive feature set.
- `label` is the target variable and must not be included among the input features.

The next step is to explicitly separate the metadata, target, numerical features, and categorical/text features.

In [4]:
# Separating metadata, target, numerical features, and categorical/text features

target_column = "label"
metadata_columns = ["FILENAME"]

# Separate target variable
y = df[target_column]

# Create feature dataframe by removing target and metadata
X = df.drop(columns=metadata_columns + [target_column])

# Identify numerical and object/text columns
numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_text_columns = X.select_dtypes(include=["object"]).columns.tolist()

print("FEATURE AND TARGET SEPARATION")
print("=" * 60)

print("Target column:", target_column)
print("Metadata column(s):", metadata_columns)
print("Total predictive features:", X.shape[1])

print("\nNumerical features:", len(numerical_columns))
print("Categorical/Text features:", len(categorical_text_columns))

print("\nCategorical/Text columns:")
for column in categorical_text_columns:
    print("-", column)

print("\nTarget shape:", y.shape)
print("Feature matrix shape:", X.shape)

FEATURE AND TARGET SEPARATION
Target column: label
Metadata column(s): ['FILENAME']
Total predictive features: 54

Numerical features: 50
Categorical/Text features: 4

Categorical/Text columns:
- URL
- Domain
- TLD
- Title

Target shape: (235795,)
Feature matrix shape: (235795, 54)


C:\Users\Hitesh\AppData\Local\Temp\ipykernel_24672\2589783156.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_text_columns = X.select_dtypes(include=["object"]).columns.tolist()


### 2.1 Checking Missing Values

Missing-value analysis is an important preprocessing step because machine learning algorithms generally require complete input data.

For this dataset, the number of missing values will be calculated for every column. This allows us to determine:

- Whether missing data is present
- Which columns contain missing values
- The number of missing values in each column
- The percentage of missing values in each column

If missing values are found, an appropriate treatment method will be selected based on the type and distribution of the corresponding feature.

If no missing values are present, no imputation will be unnecessarily applied.

In [5]:
# Missing-value analysis

missing_values = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

# Display only columns containing missing values
missing_values_present = missing_values[missing_values["Missing Values"] > 0]

print("MISSING VALUE ANALYSIS")
print("=" * 60)

if missing_values_present.empty:
    print("No missing values were found in the dataset.")
else:
    display(missing_values_present)

print("\nTotal missing values:", df.isnull().sum().sum())

MISSING VALUE ANALYSIS
No missing values were found in the dataset.

Total missing values: 0


### Interpretation of Missing-Value Analysis

The missing-value analysis shows that the dataset contains **no missing values** across its 235,795 records and 56 columns.

Therefore:

- No numerical feature requires mean or median imputation.
- No categorical feature requires mode imputation.
- No rows need to be removed because of missing values.
- The original dataset can be retained without introducing artificial values.
- This also reduces the risk of introducing bias through inappropriate imputation.

Since the dataset is complete, the preprocessing pipeline can proceed to duplicate-record analysis.

In [6]:
# Duplicate-record analysis

duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("DUPLICATE RECORD ANALYSIS")
print("=" * 60)
print("Total records:", len(df))
print("Completely duplicated rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_percentage, 2), "%")

DUPLICATE RECORD ANALYSIS
Total records: 235795
Completely duplicated rows: 0
Duplicate percentage: 0.0 %


### Interpretation of Duplicate-Record Analysis

The duplicate-record analysis shows that there are **no completely duplicated rows** in the dataset.

This means:

- Every complete record is unique based on all available columns.
- No rows need to be removed because of exact duplication.
- The original number of records remains unchanged.
- Removing rows unnecessarily could reduce the available training information.

However, exact row duplication does not necessarily mean that every URL appears only once. Therefore, URL-level duplication will also be checked separately to identify repeated URLs and understand their frequency in the dataset.

In [7]:
# URL-level duplication analysis

unique_urls = df["URL"].nunique()
repeated_url_records = len(df) - unique_urls
repeated_url_percentage = (repeated_url_records / len(df)) * 100

url_frequency = df["URL"].value_counts()
repeated_urls = url_frequency[url_frequency > 1]

print("URL-LEVEL DUPLICATION ANALYSIS")
print("=" * 60)
print("Total records:", len(df))
print("Unique URLs:", unique_urls)
print("Repeated URL records:", repeated_url_records)
print("Repeated URL percentage:", round(repeated_url_percentage, 2), "%")
print("Number of URLs appearing more than once:", len(repeated_urls))

URL-LEVEL DUPLICATION ANALYSIS
Total records: 235795
Unique URLs: 235370
Repeated URL records: 425
Repeated URL percentage: 0.18 %
Number of URLs appearing more than once: 425


### Interpretation of URL-Level Duplication

The URL-level analysis shows that the dataset contains **235,370 unique URLs** out of 235,795 total records.

A total of **425 records (0.18%)** correspond to URLs that appear more than once.

This is a very small proportion of the dataset. Therefore:

- Exact duplicate rows do not need to be removed because none were found.
- URL repetition is present but limited.
- Repeated URLs may contain different associated webpage attributes or labels and therefore should not be automatically removed without further investigation.
- The dataset retains all original records at this stage.
- The possibility of data leakage caused by repeated URLs will be considered when creating the training and testing datasets.

The next preprocessing step is to identify and examine categorical and text features before deciding how they should be handled.

In [8]:
# Identifying categorical and text features

categorical_text_columns = X.select_dtypes(include=["object"]).columns.tolist()

print("CATEGORICAL / TEXT FEATURE ANALYSIS")
print("=" * 60)

for column in categorical_text_columns:
    print(f"\nColumn: {column}")
    print("Data type:", df[column].dtype)
    print("Unique values:", df[column].nunique())
    print("Missing values:", df[column].isnull().sum())

print("\nTotal categorical/text columns:", len(categorical_text_columns))

CATEGORICAL / TEXT FEATURE ANALYSIS

Column: URL
Data type: str
Unique values: 235370
Missing values: 0

Column: Domain
Data type: str
Unique values: 220086
Missing values: 0

Column: TLD
Data type: str
Unique values: 695
Missing values: 0

Column: Title
Data type: str
Unique values: 197874
Missing values: 0

Total categorical/text columns: 4


C:\Users\Hitesh\AppData\Local\Temp\ipykernel_24672\4196610373.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_text_columns = X.select_dtypes(include=["object"]).columns.tolist()


### Interpretation of Categorical and Text Features

The dataset contains four non-numerical columns: `URL`, `Domain`, `TLD`, and `Title`.

These columns require different treatment because they do not all represent the same type of information.

- `URL` contains the complete webpage address and has very high cardinality. Using the raw URL directly as a categorical variable would create an extremely large number of categories.
- `Domain` also has high cardinality because many different domains are present. Direct one-hot encoding would create a very large feature space.
- `TLD` represents the top-level domain, such as `.com`, `.org`, or country-code domains. It has comparatively lower cardinality and can be considered for categorical encoding.
- `Title` contains webpage title information and has high cardinality. Direct categorical encoding would be inefficient.

Therefore, the raw high-cardinality text fields will not be directly one-hot encoded. Instead, useful information from URL and webpage-related text will be represented through numerical feature engineering where appropriate.

The `TLD` feature can be encoded as a categorical feature if it is retained in the final modeling pipeline.

This approach helps control dimensionality while preserving useful information for phishing URL classification.

In [9]:
# Checking the cardinality of categorical and text features

cardinality_table = pd.DataFrame({
    "Feature": categorical_text_columns,
    "Unique Values": [df[col].nunique() for col in categorical_text_columns],
    "Unique Percentage": [
        round((df[col].nunique() / len(df)) * 100, 2)
        for col in categorical_text_columns
    ]
})

print("CARDINALITY ANALYSIS")
print("=" * 60)

display(cardinality_table)

CARDINALITY ANALYSIS


,Feature,Unique Values,Unique Percentage
0,URL,235370,99.82
1,Domain,220086,93.34
2,TLD,695,0.29
3,Title,197874,83.92


### Interpretation of Cardinality Analysis

The cardinality analysis confirms that `URL`, `Domain`, and `Title` have extremely high cardinality compared with the total number of records.

The `URL` column contains 235,370 unique values, representing 99.82% of the dataset. Similarly, `Domain` contains 220,086 unique values (93.34%), while `Title` contains 197,874 unique values (83.92%).

Directly applying one-hot encoding to these high-cardinality columns would create a very large number of additional features and significantly increase memory and computational requirements.

The `TLD` column has 695 unique values, representing only 0.29% of the records. Although it has considerably lower cardinality than the other text columns, 695 categories can still produce a relatively large encoded feature set.

For this project, the raw `URL`, `Domain`, and `Title` columns will therefore not be directly one-hot encoded. Their useful information will instead be captured through numerical URL and webpage-related features already present in the dataset or through feature engineering in later notebooks.

The preprocessing pipeline will prioritize the numerical predictive features while avoiding unnecessary high-dimensional categorical encoding.

In [10]:
# Separating predictive numerical features from target and metadata

numerical_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Remove the target variable from the numerical feature list
numerical_features = [col for col in numerical_features if col != "label"]

print("NUMERICAL FEATURE ANALYSIS")
print("=" * 60)

print("Total numerical predictive features:", len(numerical_features))

print("\nNumerical predictive features:")
for i, column in enumerate(numerical_features, start=1):
    print(f"{i:2d}. {column}")

NUMERICAL FEATURE ANALYSIS
Total numerical predictive features: 50

Numerical predictive features:
 1. URLLength
 2. DomainLength
 3. IsDomainIP
 4. URLSimilarityIndex
 5. CharContinuationRate
 6. TLDLegitimateProb
 7. URLCharProb
 8. TLDLength
 9. NoOfSubDomain
10. HasObfuscation
11. NoOfObfuscatedChar
12. ObfuscationRatio
13. NoOfLettersInURL
14. LetterRatioInURL
15. NoOfDegitsInURL
16. DegitRatioInURL
17. NoOfEqualsInURL
18. NoOfQMarkInURL
19. NoOfAmpersandInURL
20. NoOfOtherSpecialCharsInURL
21. SpacialCharRatioInURL
22. IsHTTPS
23. LineOfCode
24. LargestLineLength
25. HasTitle
26. DomainTitleMatchScore
27. URLTitleMatchScore
28. HasFavicon
29. Robots
30. IsResponsive
31. NoOfURLRedirect
32. NoOfSelfRedirect
33. HasDescription
34. NoOfPopup
35. NoOfiFrame
36. HasExternalFormSubmit
37. HasSocialNet
38. HasSubmitButton
39. HasHiddenFields
40. HasPasswordField
41. Bank
42. Pay
43. Crypto
44. HasCopyrightInfo
45. NoOfImage
46. NoOfCSS
47. NoOfJS
48. NoOfSelfRef
49. NoOfEmptyRef
50. NoO

### Interpretation of Numerical Feature Identification

After separating the target variable and metadata, the dataset contains **50 numerical predictive features**.

These numerical features describe different structural and behavioral characteristics of URLs and webpages. Examples include the number of digits, special characters, links, scripts, images, CSS elements, and other webpage-related properties.

Numerical features are suitable for machine learning models because they can be represented directly as numerical input values.

However, the numerical features may have very different ranges and distributions. For example, some features may contain small counts while others may have much larger values.

Therefore, feature scaling will be considered before model training. Scaling will be performed using only the training data to prevent information from the test set from influencing the preprocessing process.

The target variable `label` remains separate and will not be scaled or used as an input feature.

In [11]:
# Creating the numerical feature matrix and splitting the dataset

# Use only numerical predictive features
X_numeric = df[numerical_features].copy()

# Target variable
y = df["label"].copy()

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_numeric,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("TRAIN-TEST SPLIT")
print("=" * 60)

print("Original feature matrix:", X_numeric.shape)
print("Training feature matrix:", X_train.shape)
print("Testing feature matrix:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

print("\nTraining proportion:", round(len(X_train) / len(X_numeric) * 100, 2), "%")
print("Testing proportion:", round(len(X_test) / len(X_numeric) * 100, 2), "%")

TRAIN-TEST SPLIT
Original feature matrix: (235795, 50)
Training feature matrix: (188636, 50)
Testing feature matrix: (47159, 50)

Training target distribution:
label
0     80756
1    107880
Name: count, dtype: int64

Testing target distribution:
label
0    20189
1    26970
Name: count, dtype: int64

Training proportion: 80.0 %
Testing proportion: 20.0 %


### Interpretation of Train-Test Split

The dataset has been divided into training and testing subsets using an **80:20 split**.

The training set is used to learn patterns and relationships between the URL-related features and the target variable. The testing set is kept separate and will be used later to evaluate how well the trained models perform on unseen data.

The `stratify=y` parameter was used to preserve the original class distribution in both subsets. This is important because the target contains two classes:

- `0` — Phishing
- `1` — Legitimate

The split was performed before feature scaling. This prevents information from the testing data from being used when calculating scaling parameters and therefore helps prevent data leakage.

The `random_state=42` parameter ensures that the same train-test split can be reproduced whenever the notebook is executed again.

The test set will remain untouched during preprocessing and model training until the final evaluation stage.

In [12]:
# Feature scaling using StandardScaler

scaler = StandardScaler()

# Fit the scaler only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same fitted scaler to transform test data
X_test_scaled = scaler.transform(X_test)

print("FEATURE SCALING")
print("=" * 60)

print("Scaler used:", "StandardScaler")
print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

print("\nScaling completed successfully.")
print("Scaler was fitted only on the training data.")

FEATURE SCALING
Scaler used: StandardScaler
Training data shape: (188636, 50)
Testing data shape: (47159, 50)

Scaling completed successfully.
Scaler was fitted only on the training data.


### Interpretation of Feature Scaling

The numerical predictive features have been standardized using the **StandardScaler** technique.

Standardization transforms each feature so that its values are centered around a mean of approximately 0 with a standard deviation of approximately 1.

The standardization formula is:

\[
z = \frac{x-\mu}{\sigma}
\]

where:

- \(x\) = original feature value
- \(\mu\) = mean of the feature calculated from the training data
- \(\sigma\) = standard deviation of the feature calculated from the training data
- \(z\) = standardized feature value

The scaler was fitted only on the training dataset using `fit_transform()`. The testing dataset was transformed using the same fitted scaler with `transform()`.

This procedure is important because calculating scaling parameters using the complete dataset could allow information from the test set to influence the training process, resulting in data leakage.

The number of features remains unchanged after scaling:

- Before scaling: 50 numerical predictive features
- After scaling: 50 numerical predictive features

Only the numerical values have been transformed; the underlying feature identities remain unchanged.

In [13]:
# Verifying the scaled training features

scaled_mean = np.mean(X_train_scaled, axis=0)
scaled_std = np.std(X_train_scaled, axis=0)

scaling_check = pd.DataFrame({
    "Feature": numerical_features,
    "Mean After Scaling": np.round(scaled_mean, 4),
    "Std After Scaling": np.round(scaled_std, 4)
})

print("SCALING VERIFICATION")
print("=" * 60)

display(scaling_check)

print("\nMaximum absolute mean:",
      round(np.max(np.abs(scaled_mean)), 6))

print("Maximum deviation of standard deviation from 1:",
      round(np.max(np.abs(scaled_std - 1)), 6))

SCALING VERIFICATION


,Feature,Mean After Scaling,Std After Scaling
0,URLLength,-0.0,1.0
1,DomainLength,-0.0,1.0
2,IsDomainIP,-0.0,1.0
3,URLSimilarityIndex,-0.0,1.0
4,CharContinuationRate,0.0,1.0
5,TLDLegitimateProb,0.0,1.0
6,URLCharProb,-0.0,1.0
7,TLDLength,-0.0,1.0
8,NoOfSubDomain,-0.0,1.0
9,HasObfuscation,0.0,1.0



Maximum absolute mean: 0.0
Maximum deviation of standard deviation from 1: 0.0


### 2.2 Checking for Constant and Near-Constant Features

Features with the same value for every observation provide no useful information for distinguishing between phishing and legitimate URLs.

A constant feature has only one unique value across the complete dataset. Such features can be removed because they do not contribute to model learning.

Near-constant features are different because they contain very little variation but are not completely constant. These features will be examined carefully rather than removed automatically.

In this step, we will first identify completely constant features and verify whether any predictive feature contains only one unique value.

If no constant predictive features are found, all numerical predictive features will be retained for subsequent analysis and feature-selection stages.

In [14]:
# Checking for constant predictive features

constant_features = [
    column for column in numerical_features
    if df[column].nunique() <= 1
]

print("CONSTANT FEATURE ANALYSIS")
print("=" * 60)

print("Number of numerical predictive features:", len(numerical_features))
print("Number of constant features:", len(constant_features))

if len(constant_features) == 0:
    print("\nNo constant predictive features were found.")
else:
    print("\nConstant features:")
    for feature in constant_features:
        print("-", feature)

CONSTANT FEATURE ANALYSIS
Number of numerical predictive features: 50
Number of constant features: 0

No constant predictive features were found.


### Interpretation of Constant Feature Analysis

The constant-feature analysis identified **no constant predictive features** among the 50 numerical features.

This means that every numerical predictive feature contains some variation across the observations and therefore has the potential to provide information to the machine learning models.

As a result:

- No numerical predictive feature needs to be removed at this stage.
- The feature matrix remains at 50 numerical predictive features.
- No information is lost through constant-feature removal.
- Further feature reduction, if required, will be performed later using feature-selection techniques based on statistical or model-based importance.

The preprocessing pipeline can now proceed to checking the overall data structure after the completed cleaning and transformation steps.

In [15]:
# Final preprocessing shape and data integrity check

print("FINAL PREPROCESSING DATA CHECK")
print("=" * 60)

print("Original dataset shape:", df.shape)
print("Predictive numerical features:", len(numerical_features))
print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

print("\nTarget training shape:", y_train.shape)
print("Target testing shape:", y_test.shape)

print("\nMissing values in training data:", np.isnan(X_train_scaled).sum())
print("Missing values in testing data:", np.isnan(X_test_scaled).sum())

print("\nConstant features removed:", len(constant_features))

print("\nPreprocessing integrity check completed.")

FINAL PREPROCESSING DATA CHECK
Original dataset shape: (235795, 56)
Predictive numerical features: 50
Training data shape: (188636, 50)
Testing data shape: (47159, 50)

Target training shape: (188636,)
Target testing shape: (47159,)

Missing values in training data: 0
Missing values in testing data: 0

Constant features removed: 0

Preprocessing integrity check completed.


### Interpretation of Final Preprocessing Check

The final preprocessing check confirms that the data preparation process has been completed successfully.

The original dataset contains **235,795 records and 56 columns**. After excluding the metadata column `FILENAME` and the target variable `label`, the predictive dataset contains **50 numerical features** selected for the current modeling pipeline.

The data was divided into:

- **188,636 training records (80%)**
- **47,159 testing records (20%)**
- **50 numerical predictive features**

The integrity check also confirms that:

- No missing values are present in the scaled training data.
- No missing values are present in the scaled testing data.
- No constant predictive features were identified.
- The target variable remains separate from the feature matrices.
- Scaling was performed using parameters learned only from the training data.

Therefore, the preprocessing pipeline has produced clean and standardized training and testing datasets suitable for machine learning model development.

The next step is to verify that the target variable has not been accidentally included in the feature matrix and that no obvious data leakage exists between the feature and target sets.

In [16]:
# Verifying target separation and checking for data leakage

print("TARGET SEPARATION AND DATA LEAKAGE CHECK")
print("=" * 60)

# Check whether the target variable is present in the feature list
target_in_features = "label" in numerical_features

# Check whether training and testing indices overlap
train_test_overlap = len(set(X_train.index).intersection(set(X_test.index)))

# Check whether the target values are stored separately
target_separated = (
    "label" not in X_train.columns
    and "label" not in X_test.columns
)

print("Target variable included in features:", target_in_features)
print("Target variable separated from features:", target_separated)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))
print("Overlapping records between training and testing:", train_test_overlap)

print("\nData leakage check:")

if not target_in_features and target_separated and train_test_overlap == 0:
    print("No obvious target or train-test leakage was detected.")
else:
    print("Potential data leakage detected. Further investigation is required.")

TARGET SEPARATION AND DATA LEAKAGE CHECK
Target variable included in features: False
Target variable separated from features: True

Training records: 188636
Testing records: 47159
Overlapping records between training and testing: 0

Data leakage check:
No obvious target or train-test leakage was detected.


### Interpretation of Target Separation and Initial Leakage Check

The target-separation check confirms that the `label` variable is not included among the predictive features.

The training and testing datasets also contain no overlapping row indices, confirming that the same dataset records were not directly placed in both subsets.

The preprocessing pipeline therefore maintains a clear separation between:

- Input features (`X`)
- Target variable (`y`)
- Training data
- Testing data

However, because the original dataset contains a small number of repeated URLs, an additional URL-level overlap check is required. This is important because records containing the same URL could potentially appear in both the training and testing datasets even when their row indices are different.

Therefore, the next step will explicitly compare the URLs in the training and testing subsets to provide stronger evidence regarding URL-level data leakage.

In [17]:
# Checking URL-level overlap between training and testing datasets

train_urls = set(df.loc[X_train.index, "URL"])
test_urls = set(df.loc[X_test.index, "URL"])

common_urls = train_urls.intersection(test_urls)

print("URL-LEVEL DATA LEAKAGE CHECK")
print("=" * 60)

print("Unique URLs in training set:", len(train_urls))
print("Unique URLs in testing set:", len(test_urls))
print("URLs appearing in both sets:", len(common_urls))

if len(common_urls) == 0:
    print("\nNo URL-level overlap was detected between training and testing data.")
else:
    print("\nURL-level overlap detected.")
    print("These repeated URLs require consideration when interpreting model performance.")

URL-LEVEL DATA LEAKAGE CHECK
Unique URLs in training set: 188370
Unique URLs in testing set: 47143
URLs appearing in both sets: 143

URL-level overlap detected.
These repeated URLs require consideration when interpreting model performance.


### Interpretation of URL-Level Data Leakage Check

The URL-level leakage check identified **143 URLs that appear in both the training and testing datasets**.

This occurs because the dataset contains a small number of repeated URLs, and the initial random train-test split was performed at the record level rather than at the URL-group level.

Although the repeated URLs represent only a small portion of the complete dataset, allowing the same URL to occur in both training and testing data can result in overly optimistic evaluation because the model may encounter a URL-related pattern during testing that was already present during training.

Therefore, the initial random split will not be used for the final model evaluation.

To prevent URL-level leakage, the dataset will be re-split using a **group-based stratified splitting strategy**, where all records belonging to the same URL are kept in the same subset.

This ensures that:

- No URL is shared between training and testing data.
- The target class distribution remains approximately balanced.
- The testing set represents URLs that are unseen during training.
- Model performance can be evaluated more reliably.

The corrected leakage-free train-test split will be used for the remainder of the machine learning pipeline.

In [18]:
# Creating a leakage-free train-test split based on unique URLs

from sklearn.model_selection import train_test_split

# Create one record for each unique URL
url_groups = df[["URL", "label"]].drop_duplicates(subset="URL")

# Split unique URLs while preserving the target distribution
train_url_groups, test_url_groups = train_test_split(
    url_groups,
    test_size=0.20,
    random_state=42,
    stratify=url_groups["label"]
)

# Get the URLs assigned to each subset
train_url_set = set(train_url_groups["URL"])
test_url_set = set(test_url_groups["URL"])

# Create masks for the original dataset
train_mask = df["URL"].isin(train_url_set)
test_mask = df["URL"].isin(test_url_set)

# Create leakage-free feature and target datasets
X_train = X_numeric.loc[train_mask].copy()
X_test = X_numeric.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("LEAKAGE-FREE TRAIN-TEST SPLIT")
print("=" * 60)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining unique URLs:", X_train.shape[0])
print("Testing unique URLs:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

print("\nTraining proportion:",
      round(len(X_train) / len(df) * 100, 2), "%")

print("Testing proportion:",
      round(len(X_test) / len(df) * 100, 2), "%")

LEAKAGE-FREE TRAIN-TEST SPLIT
Training records: 188644
Testing records: 47151

Training unique URLs: 188644
Testing unique URLs: 47151

Training target distribution:
label
0     80764
1    107880
Name: count, dtype: int64

Testing target distribution:
label
0    20181
1    26970
Name: count, dtype: int64

Training proportion: 80.0 %
Testing proportion: 20.0 %


### Verification of the Leakage-Free Split

The dataset has now been re-split at the **URL-group level** rather than only at the individual-record level.

This approach ensures that all records belonging to the same URL are assigned to the same subset. Therefore, a URL that is present in the training dataset should not appear in the testing dataset.

The new split will be verified for:

- URL overlap between training and testing datasets
- Training and testing record counts
- Target-class distribution
- Preservation of the intended approximate 80:20 split

This verification is important because the leakage-free split will be used for all subsequent preprocessing, model training, and final evaluation.

Only after this verification will the scaler be fitted again on the corrected training data.

In [19]:
# Verifying the leakage-free train-test split

train_urls = set(df.loc[X_train.index, "URL"])
test_urls = set(df.loc[X_test.index, "URL"])

common_urls = train_urls.intersection(test_urls)

print("LEAKAGE-FREE SPLIT VERIFICATION")
print("=" * 60)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining unique URLs:", len(train_urls))
print("Testing unique URLs:", len(test_urls))

print("\nURLs appearing in both sets:", len(common_urls))

print("\nTraining phishing records:", (y_train == 0).sum())
print("Training legitimate records:", (y_train == 1).sum())

print("Testing phishing records:", (y_test == 0).sum())
print("Testing legitimate records:", (y_test == 1).sum())

if len(common_urls) == 0:
    print("\nSUCCESS: No URL-level overlap exists between training and testing data.")
else:
    print("\nWARNING: URL-level overlap still exists.")

LEAKAGE-FREE SPLIT VERIFICATION
Training records: 188644
Testing records: 47151

Training unique URLs: 188296
Testing unique URLs: 47074

URLs appearing in both sets: 0

Training phishing records: 80764
Training legitimate records: 107880
Testing phishing records: 20181
Testing legitimate records: 26970

SUCCESS: No URL-level overlap exists between training and testing data.


### Interpretation of Leakage-Free Split Verification

The verification step confirms that the training and testing datasets were created using URL-level grouping.

The most important condition is that **no URL should occur in both the training and testing subsets**. This prevents repeated URLs from providing the model with information in the training data that could reappear during testing.

The corrected split also maintains separate training and testing records and preserves the target classes `0` (phishing) and `1` (legitimate) through stratified splitting at the unique-URL level.

This URL-group-based splitting strategy provides a more reliable evaluation setup than the initial record-level random split.

The corrected training and testing datasets will now be used for the remaining preprocessing and machine learning stages.

> Note: The exact training/testing record counts and class proportions should be taken from the output of the verification cell rather than assumed in advance.

In [20]:
# Re-scaling the corrected leakage-free train-test split

scaler = StandardScaler()

# Fit only on the corrected training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform testing data using the training-fitted scaler
X_test_scaled = scaler.transform(X_test)

print("LEAKAGE-FREE FEATURE SCALING")
print("=" * 60)

print("Scaler used:", "StandardScaler")
print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

print("\nScaler fitted only on leakage-free training data.")
print("Testing data transformed using the same fitted scaler.")
print("\nScaling completed successfully.")

LEAKAGE-FREE FEATURE SCALING
Scaler used: StandardScaler
Training data shape: (188644, 50)
Testing data shape: (47151, 50)

Scaler fitted only on leakage-free training data.
Testing data transformed using the same fitted scaler.

Scaling completed successfully.


### Interpretation of Final Feature Scaling

The corrected URL-level train-test split has now been standardized using `StandardScaler`.

The scaler was fitted exclusively on the leakage-free training dataset. The same fitted scaler was then used to transform the testing dataset.

This ensures that:

- The testing data does not influence the calculation of the scaling parameters.
- The training and testing features have the same numerical scale.
- The original 50 numerical predictive features are retained.
- No additional features are created or removed during scaling.
- The preprocessing process remains reproducible because the scaler is fitted from a fixed training split.

The resulting `X_train_scaled` and `X_test_scaled` matrices are now ready for machine learning model development.

The target arrays `y_train` and `y_test` remain unchanged because the target variable represents class labels rather than continuous numerical features.

In [21]:
# Final verification of scaled features

print("FINAL SCALING VERIFICATION")
print("=" * 60)

train_means = np.mean(X_train_scaled, axis=0)
train_stds = np.std(X_train_scaled, axis=0)

print("Maximum absolute training mean:",
      round(np.max(np.abs(train_means)), 6))

print("Maximum deviation of training standard deviation from 1:",
      round(np.max(np.abs(train_stds - 1)), 6))

print("\nTraining shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

print("\nMissing values in training:",
      np.isnan(X_train_scaled).sum())

print("Missing values in testing:",
      np.isnan(X_test_scaled).sum())

print("\nFinal scaling verification completed.")

FINAL SCALING VERIFICATION
Maximum absolute training mean: 0.0
Maximum deviation of training standard deviation from 1: 0.0

Training shape: (188644, 50)
Testing shape: (47151, 50)

Missing values in training: 0
Missing values in testing: 0

Final scaling verification completed.


### Interpretation of Final Scaling Verification

The final verification confirms that the corrected training features have been standardized successfully.

The mean of the scaled training features is approximately 0, while their standard deviation is approximately 1. This confirms that `StandardScaler` has been applied correctly.

The verification also confirms that:

- The training and testing datasets contain the same 50 predictive features.
- No missing values were introduced during scaling.
- The target variable is kept separate from the feature matrices.
- The scaler was fitted only on the leakage-free training data.
- The same scaler was used to transform the testing data.

Therefore, the numerical preprocessing stage is complete and the resulting datasets are ready for machine learning model development.

In [22]:
# Final preprocessing summary

print("FINAL PREPROCESSING SUMMARY")
print("=" * 70)

print("Original dataset:")
print("  Records:", len(df))
print("  Total columns:", len(df.columns))

print("\nRemoved from predictive features:")
print("  Metadata column: FILENAME")
print("  Target column: label")

print("\nFinal predictive features:")
print("  Numerical features:", len(numerical_features))
print("  Constant features removed:", len(constant_features))

print("\nFinal dataset split:")
print("  Training records:", len(X_train_scaled))
print("  Testing records:", len(X_test_scaled))
print("  Training features:", X_train_scaled.shape[1])
print("  Testing features:", X_test_scaled.shape[1])

print("\nData quality:")
print("  Missing values: 0")
print("  Exact duplicate rows: 0")
print("  URL overlap between train/test:", len(common_urls))

print("\nPreprocessing techniques:")
print("  Missing-value treatment: Not required")
print("  Duplicate treatment: No exact duplicates found")
print("  High-cardinality text handling: Raw text excluded from current model matrix")
print("  Feature scaling: StandardScaler")
print("  Data leakage prevention: URL-level grouped split")

print("\n" + "=" * 70)
print("DATA PREPROCESSING COMPLETED SUCCESSFULLY")

FINAL PREPROCESSING SUMMARY
Original dataset:
  Records: 235795
  Total columns: 56

Removed from predictive features:
  Metadata column: FILENAME
  Target column: label

Final predictive features:
  Numerical features: 50
  Constant features removed: 0

Final dataset split:
  Training records: 188644
  Testing records: 47151
  Training features: 50
  Testing features: 50

Data quality:
  Missing values: 0
  Exact duplicate rows: 0
  URL overlap between train/test: 0

Preprocessing techniques:
  Missing-value treatment: Not required
  Duplicate treatment: No exact duplicates found
  High-cardinality text handling: Raw text excluded from current model matrix
  Feature scaling: StandardScaler
  Data leakage prevention: URL-level grouped split

DATA PREPROCESSING COMPLETED SUCCESSFULLY


### Final Interpretation of Data Preprocessing

The data preprocessing stage has prepared the phishing URL dataset for subsequent exploratory analysis and machine learning model development.

The preprocessing workflow included:

1. Loading and verifying the original dataset.
2. Separating the target variable `label` from the predictive features.
3. Excluding the metadata column `FILENAME`.
4. Checking for missing values and confirming that none were present.
5. Checking for exact duplicate records and confirming that none were present.
6. Examining URL-level duplication.
7. Identifying high-cardinality text features.
8. Retaining the numerical predictive features for the current modeling pipeline.
9. Identifying and checking for constant features.
10. Creating a URL-level leakage-free train-test split.
11. Standardizing the numerical features using `StandardScaler`.
12. Fitting the scaler only on the training data and applying it to the testing data.
13. Verifying the final training and testing datasets for missing values and URL overlap.

The final preprocessing pipeline is designed to reduce unnecessary dimensionality and prevent information from the testing dataset from influencing model training.

The processed training and testing datasets are now ready for the next stage of the project: **Exploratory Data Analysis (EDA)**.

In [23]:
# Saving the final preprocessed datasets and scaler

import os
import joblib

# Create a folder for processed machine learning data
processed_path = "../dataset/processed"
os.makedirs(processed_path, exist_ok=True)

# Save scaled training/testing features and targets
np.savez(
    os.path.join(processed_path, "preprocessed_data.npz"),
    X_train_scaled=X_train_scaled,
    X_test_scaled=X_test_scaled,
    y_train=y_train.to_numpy(),
    y_test=y_test.to_numpy()
)

# Save the StandardScaler
joblib.dump(
    scaler,
    os.path.join(processed_path, "standard_scaler.pkl")
)

# Save the feature names
with open(
    os.path.join(processed_path, "feature_names.txt"),
    "w"
) as file:
    for feature in numerical_features:
        file.write(feature + "\n")

print("PREPROCESSED DATA SAVED SUCCESSFULLY")
print("=" * 60)

print("Saved folder:", processed_path)
print("\nFiles created:")
print("- preprocessed_data.npz")
print("- standard_scaler.pkl")
print("- feature_names.txt")

print("\nTraining data saved:", X_train_scaled.shape)
print("Testing data saved:", X_test_scaled.shape)
print("Number of feature names saved:", len(numerical_features))

print("\nThese files can be reused in later machine learning notebooks.")

PREPROCESSED DATA SAVED SUCCESSFULLY
Saved folder: ../dataset/processed

Files created:
- preprocessed_data.npz
- standard_scaler.pkl
- feature_names.txt

Training data saved: (188644, 50)
Testing data saved: (47151, 50)
Number of feature names saved: 50

These files can be reused in later machine learning notebooks.


# Conclusion of Data Preprocessing

The data preprocessing stage for the Phishing URL Detection project has been completed successfully.

The original dataset contained **235,795 records and 56 columns**. The metadata column `FILENAME` and target variable `label` were separated from the predictive feature set.

The dataset was examined for data-quality issues. No missing values and no completely duplicated rows were found. URL-level duplication was also investigated because repeated URLs can affect the reliability of model evaluation.

A URL-level train-test splitting strategy was implemented to ensure that the same URL does not appear in both training and testing datasets. This reduces the risk of URL-level data leakage.

The dataset contains **50 numerical predictive features** after excluding the target variable. High-cardinality textual fields such as `URL`, `Domain`, and `Title` were not directly one-hot encoded because doing so would create an unnecessarily large feature space. `TLD` was also not included in the current numerical modeling matrix.

The numerical features were standardized using **StandardScaler**. The scaler was fitted only on the training data and subsequently used to transform the testing data.

The final preprocessing pipeline therefore provides:

- Clean numerical input features
- Separate target labels
- Leakage-free URL-level train-test separation
- Standardized numerical features
- No missing values in the processed data
- No constant predictive features
- Reusable preprocessing files for subsequent notebooks

The processed datasets and scaler have been saved for use in the remaining machine learning workflow.

**Notebook 02 — Data Preprocessing is now complete.**

The next stage of the project is:

**Notebook 03 — Exploratory Data Analysis (EDA)**

In the EDA stage, the dataset will be statistically analyzed and visualized to identify patterns, distributions, relationships, correlations, and differences between phishing and legitimate URLs.